# 📊 Lección 4 – Procesamiento con Spark SQL y DataFrames
### Proyecto: Retail Analytics Pipeline — RetailMax
**Módulo 9: Fundamentos de Big Data | Alkemy**

---
**Objetivo:** Transformar RDDs en DataFrames, aplicar esquemas explícitos, ejecutar consultas SQL para generar métricas de negocio y guardar resultados en formato Parquet.

In [ ]:
# ============================================================
# Setup
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    FloatType, ArrayType
)
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns

spark = (
    SparkSession.builder
    .appName('RetailMax_L4_DataFrames')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print(f'✅ SparkSession lista | Spark {sc.version}')

In [ ]:
# ============================================================
# 1. Cargar CSV directamente como DataFrame con schema explícito
# ============================================================
DATA_PATH = '../data/fashion_mnist'

# Schema explícito para las primeras columnas + píxeles
# (Más eficiente que inferSchema=True — evita escanear el archivo completo)
pixel_fields = [StructField(f'pixel_{i}', IntegerType(), True) for i in range(784)]

schema = StructType([
    StructField('label',      IntegerType(), False),
    StructField('label_name', StringType(),  False),
] + pixel_fields)

df_train = (
    spark.read
    .option('header', 'true')
    .schema(schema)
    .csv(os.path.join(DATA_PATH, 'fashion_train.csv'))
)

df_test = (
    spark.read
    .option('header', 'true')
    .schema(schema)
    .csv(os.path.join(DATA_PATH, 'fashion_test.csv'))
)

df_train.cache()
df_test.cache()

print(f'Train rows: {df_train.count():,} | Columns: {len(df_train.columns)}')
print(f'Test  rows: {df_test.count():,}  | Columns: {len(df_test.columns)}')
df_train.select('label', 'label_name', 'pixel_0', 'pixel_391', 'pixel_783').show(5)

In [ ]:
# ============================================================
# 2. Enriquecer el DataFrame con métricas de negocio
# ============================================================
# Calcular intensidad media y varianza de cada imagen
pixel_cols = [f'pixel_{i}' for i in range(784)]

# Crear columnas derivadas
df_enriched = df_train.select(
    'label',
    'label_name',
    *pixel_cols
).withColumn(
    'intensidad_media',
    sum([F.col(p) for p in pixel_cols]) / 784
).withColumn(
    'pixel_max',  F.greatest(*[F.col(p) for p in pixel_cols])
).withColumn(
    'pixel_min',  F.least(*[F.col(p) for p in pixel_cols])
).withColumn(
    'contraste',  F.col('pixel_max') - F.col('pixel_min')
).withColumn(
    'segmento_brillo',
    F.when(F.col('intensidad_media') < 64,  'oscuro')
     .when(F.col('intensidad_media') < 128, 'medio')
     .otherwise('claro')
)

# Solo columnas relevantes (sin los 784 píxeles) para consultas SQL
df_metrics = df_enriched.select(
    'label', 'label_name', 'intensidad_media', 
    'pixel_max', 'pixel_min', 'contraste', 'segmento_brillo'
)

df_metrics.cache()
df_metrics.show(5)
print(f'Schema del DataFrame enriquecido:')
df_metrics.printSchema()

In [ ]:
# ============================================================
# 3. Registrar tabla temporal para Spark SQL
# ============================================================
df_metrics.createOrReplaceTempView('productos')
df_train.select('label', 'label_name').createOrReplaceTempView('catalogo_raw')

print('✅ Vistas temporales registradas:')
spark.sql('SHOW TABLES').show()

In [ ]:
# ============================================================
# SPARK SQL — Consulta 1: Ventas/conteo por categoría
# ============================================================
q1 = spark.sql("""
    SELECT 
        label_name                          AS categoria,
        COUNT(*)                            AS total_productos,
        ROUND(AVG(intensidad_media), 2)     AS brillo_promedio,
        ROUND(AVG(contraste), 2)            AS contraste_promedio,
        ROUND(MIN(intensidad_media), 2)     AS brillo_minimo,
        ROUND(MAX(intensidad_media), 2)     AS brillo_maximo
    FROM productos
    GROUP BY label_name
    ORDER BY total_productos DESC
""")

print('=== CONSULTA 1: Métricas por categoría ===')
q1.show(truncate=False)

In [ ]:
# ============================================================
# SPARK SQL — Consulta 2: Top productos (mayor contraste → más detalle visual)
# ============================================================
q2 = spark.sql("""
    SELECT 
        label_name,
        contraste,
        intensidad_media,
        segmento_brillo,
        RANK() OVER (PARTITION BY label_name ORDER BY contraste DESC) AS rank_contraste
    FROM productos
    QUALIFY rank_contraste <= 3
    ORDER BY label_name, rank_contraste
""")
# Nota: QUALIFY no está soportado en todas las versiones, alternativa:
q2_alt = spark.sql("""
    SELECT label_name, ROUND(AVG(contraste),2) as contraste_medio,
           ROUND(PERCENTILE_APPROX(contraste, 0.75),2) as contraste_p75
    FROM productos
    GROUP BY label_name
    ORDER BY contraste_medio DESC
""")

print('=== CONSULTA 2: Top categorías por contraste visual (detalle) ===')
q2_alt.show(truncate=False)

In [ ]:
# ============================================================
# SPARK SQL — Consulta 3: Segmentación de brillo para marketing
# ============================================================
q3 = spark.sql("""
    SELECT 
        label_name AS categoria,
        segmento_brillo,
        COUNT(*) AS cantidad,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY label_name), 1) AS pct_categoria
    FROM productos
    GROUP BY label_name, segmento_brillo
    ORDER BY label_name, segmento_brillo
""")

print('=== CONSULTA 3: Segmentación de brillo por categoría (para marketing) ===')
q3.show(30, truncate=False)

In [ ]:
# ============================================================
# SPARK SQL — Consulta 4: Correlación entre brillo y contraste
# ============================================================
q4 = spark.sql("""
    SELECT 
        label_name,
        ROUND(CORR(intensidad_media, contraste), 4) AS correlacion_brillo_contraste,
        ROUND(STDDEV(intensidad_media), 4) AS variabilidad_brillo
    FROM productos
    GROUP BY label_name
    ORDER BY correlacion_brillo_contraste DESC
""")

print('=== CONSULTA 4: Correlación brillo-contraste (consistencia visual del producto) ===')
q4.show(truncate=False)

In [ ]:
# ============================================================
# Visualización de métricas de negocio
# ============================================================
import pandas as pd

df_q1_pd = q1.toPandas()
df_q3_pd = q3.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('RetailMax — Métricas de Negocio Fashion-MNIST\n(Spark SQL + DataFrames)', 
             fontsize=14, fontweight='bold')

# 1. Brillo promedio por categoría
colors = plt.cm.RdYlGn(df_q1_pd['brillo_promedio'] / 255)
axes[0,0].barh(df_q1_pd['categoria'], df_q1_pd['brillo_promedio'], color=colors)
axes[0,0].set_title('Brillo Promedio por Categoría', fontweight='bold')
axes[0,0].set_xlabel('Intensidad media (0-255)')

# 2. Contraste promedio
df_q1_sort = df_q1_pd.sort_values('contraste_promedio', ascending=True)
axes[0,1].barh(df_q1_sort['categoria'], df_q1_sort['contraste_promedio'], 
               color=plt.cm.Blues(df_q1_sort['contraste_promedio']/255))
axes[0,1].set_title('Contraste Visual Promedio por Categoría', fontweight='bold')
axes[0,1].set_xlabel('Contraste (max_px - min_px)')

# 3. Segmentación de brillo — heatmap
pivot = df_q3_pd.pivot_table(values='pct_categoria', index='categoria', 
                              columns='segmento_brillo', aggfunc='sum').fillna(0)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1,0],
            cbar_kws={'label': '% de la categoría'})
axes[1,0].set_title('% de Brillo por Categoría (Segmentación Marketing)', fontweight='bold')
axes[1,0].set_xlabel('Segmento de brillo')

# 4. Scatter brillo vs contraste
axes[1,1].scatter(df_q1_pd['brillo_promedio'], df_q1_pd['contraste_promedio'], 
                  s=df_q1_pd['total_productos']/20, alpha=0.7,
                  c=range(len(df_q1_pd)), cmap='tab10')
for _, row in df_q1_pd.iterrows():
    axes[1,1].annotate(row['categoria'], (row['brillo_promedio'], row['contraste_promedio']),
                       fontsize=7, xytext=(3,3), textcoords='offset points')
axes[1,1].set_xlabel('Brillo promedio')
axes[1,1].set_ylabel('Contraste promedio')
axes[1,1].set_title('Mapa Brillo vs Contraste (tamaño = # productos)', fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../visualizations', exist_ok=True)
plt.savefig('../visualizations/L4_metricas_negocio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización guardada: visualizations/L4_metricas_negocio.png')

In [ ]:
# ============================================================
# Guardar resultados en formato PARQUET
# ============================================================
OUTPUTS_PATH = '../outputs'
os.makedirs(OUTPUTS_PATH, exist_ok=True)

# 1. DataFrame de métricas → Parquet (para MLlib en L5)
parquet_path = os.path.join(OUTPUTS_PATH, 'fashion_metrics.parquet')
df_metrics.write.mode('overwrite').parquet(parquet_path)
print(f'✅ Parquet guardado: {parquet_path}')

# 2. DataFrame completo (con píxeles) → Parquet (para features de ML)
# Solo guardamos una muestra de 10k para mantener tamaño manejable
df_ml = df_enriched.sample(fraction=0.167, seed=42)  # ~10k registros
ml_parquet = os.path.join(OUTPUTS_PATH, 'fashion_ml_features.parquet')
df_ml.write.mode('overwrite').parquet(ml_parquet)
print(f'✅ Parquet ML guardado: {ml_parquet}')

# 3. Métricas por categoría → CSV para reporte
csv_path = os.path.join(OUTPUTS_PATH, 'metricas_por_categoria.csv')
q1.toPandas().to_csv(csv_path, index=False)
print(f'✅ CSV guardado: {csv_path}')

# Verificar archivos generados
print(f'\nArchivos en {OUTPUTS_PATH}:')
for f in os.listdir(OUTPUTS_PATH):
    print(f'  {f}')

In [ ]:
# ============================================================
# Verificar que el Parquet se puede leer correctamente
# ============================================================
df_verify = spark.read.parquet(parquet_path)
print('=== VERIFICACIÓN DEL PARQUET ===')
print(f'Rows: {df_verify.count():,} | Cols: {len(df_verify.columns)}')
df_verify.show(3)
print('✅ Parquet leído correctamente. Listo para Lección 5 (MLlib).')

---
## ✅ Checklist Lección 4
- [x] RDDs transformados en DataFrames con schema explícito
- [x] Columnas derivadas calculadas: `intensidad_media`, `contraste`, `segmento_brillo`
- [x] Vista temporal registrada y consultada con Spark SQL
- [x] 4 consultas SQL ejecutadas: métricas por categoría, top contraste, segmentación, correlación
- [x] Resultados guardados en formato Parquet (para MLlib)
- [x] Métricas exportadas como CSV
- [x] Visualizaciones generadas

**Próximo paso → Lección 5:** Pipeline de Machine Learning con Spark MLlib.